# Vehicle Type Recognition from Images
## Part 2 — Convolutional Modelling and Evaluation

**Dataset:** VTID2 (Vehicle Type Image Dataset 2)
**Author:** Mohammed Mahyoub Ali Ayedh Mohammed — MSc Artificial Intelligence, UWE Bristol

This notebook consumes `results/split_manifest.csv`, produced by
`01_data_audit_and_leakage_control.ipynb`. It does not repeat any duplicate or
leakage analysis: that work is complete, and its result is treated here as a fixed
input.

# 1. Objective

Two convolutional models are developed and compared on the leakage-controlled
partition:

1. a network built from convolutional components alone, trained from random
   initialisation, which establishes what the data supports without external
   knowledge;
2. a ResNet-18 adapted by transfer learning, which establishes what is achievable
   when representations learned from a large corpus are reused.

Both are evaluated once on the held-out test partition, after all architectural and
hyperparameter decisions have been made on the validation partition. Section 10 then
uses the same architecture to measure how much the leakage control of Part 1 was
worth, by repeating the experiment on a partition that ignores it.

Reporting a single accuracy figure would say little. The comparison of interest is
between models, between partitions, and between what each model appears to have
learned.

# 2. Setup

In [ ]:
import sys
import time
from copy import deepcopy
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

import config as cfg
import vtid_audit as va

va.set_seed(cfg.RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
print(f"Device      : {device}")
if device.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

# 3. The Leakage-Controlled Partition

The manifest records, for every retained image, its class, its leakage-risk group and
its partition. The check below re-confirms that no group spans a partition boundary.
It duplicates a check already made in Part 1, deliberately: this notebook is the one
that produces the accuracy figures, so it should not rely on an assurance made in a
file it does not execute.

In [ ]:
manifest = pd.read_csv(cfg.SPLIT_MANIFEST)

spanning = manifest.groupby("group_id")["split"].nunique().gt(1).sum()
print(f"Images                          : {len(manifest):,}")
print(f"Leakage-risk groups             : {manifest['group_id'].nunique():,}")
print(f"Groups spanning partitions      : {spanning}")
assert spanning == 0, "Partition integrity check failed"

class_names = [cfg.CLASS_DISPLAY[name] for name in cfg.CLASS_ORDER]
class_index = {name: index for index, name in enumerate(cfg.CLASS_ORDER)}
manifest["label"] = manifest["class_name"].map(class_index)

va.split_composition(manifest, cfg.CLASS_DISPLAY)

# 4. Preprocessing and Augmentation

## 4.1 Deterministic preprocessing

Images vary in size, so all are resized to 224 x 224 and normalised with the ImageNet
channel statistics. The resolution matches the ResNet-18 input and is small enough to
train on 4 GB of video memory; the normalisation is required for the pretrained model
and is applied to the scratch model as well so that the two differ only in
architecture and initialisation.

## 4.2 Augmentation is applied to the training partition only

Augmenting before partitioning would place a transformed copy of an image in one
partition and its original in another, which is the same leakage the audit was built
to prevent. Augmentation is therefore applied inside the training `DataLoader` and
nowhere else; validation and test use the deterministic pipeline.

## 4.3 Choosing transformations that preserve the label

Each transformation is included only if it produces an image that a human would still
assign to the same vehicle type.

| Transformation | Included | Reasoning |
|---|---|---|
| Horizontal flip | Yes | A mirrored vehicle is the same type; both orientations occur naturally in traffic imagery |
| Vertical flip | **No** | Produces images that cannot occur and that share no statistics with the test distribution |
| Small rotation (±10°) | Yes | Reflects camera tilt |
| Colour jitter | Yes | Paint colour and lighting do not define a vehicle type; discouraging reliance on them is desirable |
| Random resized crop (scale 0.8–1.0) | Yes, conservatively | Reflects framing variation, but an aggressive crop can remove the load bed that distinguishes a pickup from an SUV, which would corrupt the label |
| Grayscale conversion | **No** | Discards a channel the model may legitimately use, without a corresponding gain |

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.RandomResizedCrop(cfg.IMAGE_SIZE, scale=(0.8, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
])


class VehicleDataset(Dataset):
    """Images and labels for one partition of the manifest."""

    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].to_numpy()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, int(self.labels[index])


def build_loaders(frame, batch_size=32, num_workers=0):
    partitions = {name: frame[frame["split"] == name] for name in ("train", "val", "test")}
    return {
        name: DataLoader(
            VehicleDataset(part, train_transform if name == "train" else eval_transform),
            batch_size=batch_size,
            shuffle=(name == "train"),
            num_workers=num_workers,
            drop_last=False,
        )
        for name, part in partitions.items()
    }


loaders = build_loaders(manifest)
print({name: len(loader.dataset) for name, loader in loaders.items()})

# 5. Class Imbalance

The classes are unequally represented, with roughly three and a half times as many
pickups as SUVs. Left unaddressed, a network minimising average error can improve its
loss by under-predicting the smaller classes, and overall accuracy will conceal that
behaviour.

The loss is therefore weighted by inverse class frequency. The weights are computed
from the **training partition only**. Deriving them from the whole dataset would let a
property of the validation and test sets influence training — a small leak, but the
same kind the audit exists to prevent.

Weighting is preferred here to oversampling because oversampling repeats individual
images within an epoch, which interacts badly with a dataset that has just been
deduplicated: it would reintroduce exact repetition by a different route. Macro-averaged
F1 is used for model selection throughout, since it weights each class equally
regardless of size.

In [ ]:
train_frame = manifest[manifest["split"] == "train"]
counts = train_frame["label"].value_counts().sort_index().to_numpy()

class_weights = torch.tensor(
    counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device
)

pd.DataFrame({
    "Class": class_names,
    "Training images": counts,
    "Loss weight": class_weights.cpu().numpy().round(3),
}).set_index("Class")

# 6. A Convolutional Network Built from Its Components

The first model is assembled from the operations that define a convolutional network,
so that its behaviour can be attributed to specific design choices rather than to a
published architecture.

Each block applies two 3 x 3 convolutions, each followed by batch normalisation and a
ReLU activation, and ends with 2 x 2 max pooling. Stacking two small convolutions
before pooling widens the receptive field while using fewer parameters than a single
larger kernel, and the intermediate non-linearity makes the pair strictly more
expressive than one convolution of equivalent extent. Channel width doubles as
spatial resolution halves, which keeps the computational cost per block roughly
constant while the representation moves from local edges towards object-level
structure.

Two choices deserve explicit justification:

- **Batch normalisation before the activation** stabilises the input distribution of
  each layer, which permits a higher learning rate and shortens training. On a dataset
  of a few thousand images this is what makes training from random initialisation
  practical at all.
- **Global average pooling instead of a flattened fully connected layer** reduces the
  classifier to one weight per channel per class. A flattening layer at this
  resolution would introduce several million parameters — more than the rest of the
  network combined — and, with roughly three thousand training images, is the most
  likely single cause of overfitting.

In [ ]:
class ConvBlock(nn.Module):
    """Two 3x3 convolutions with batch normalisation, then 2x2 max pooling."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class VehicleCNN(nn.Module):
    """A compact convolutional classifier trained from random initialisation."""

    def __init__(self, n_classes, widths=(32, 64, 128, 256), dropout=0.4):
        super().__init__()
        channels = [3, *widths]
        self.features = nn.Sequential(
            *[ConvBlock(channels[i], channels[i + 1]) for i in range(len(widths))]
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(widths[-1], n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


scratch_model = VehicleCNN(len(class_names)).to(device)

parameters = sum(p.numel() for p in scratch_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {parameters:,}")
print(f"Training images     : {len(train_frame):,}")
print(f"Parameters per training image: {parameters / len(train_frame):,.0f}")

The final ratio is worth stating rather than hiding: the network has far more
parameters than training images. That is normal for convolutional models and is not
by itself fatal, because weight sharing means the effective capacity is much lower
than the parameter count suggests. It does make regularisation — augmentation,
dropout, weight decay and early stopping — load-bearing rather than decorative, and
it is the main reason the transfer-learned model in Section 8 is expected to do
better.

# 7. Training Protocol

The same procedure trains both models, so that any difference between them is
attributable to the model rather than the schedule.

- **Loss:** class-weighted cross-entropy.
- **Optimiser:** AdamW, which decouples weight decay from the gradient update.
- **Schedule:** cosine annealing over the epoch budget.
- **Model selection:** the parameters achieving the best macro F1 on the validation
  partition are retained; training stops after a fixed number of epochs without
  improvement.

The test partition is not read anywhere in this section. Selecting an epoch, a
learning rate or an architecture using test performance would make the final figure a
training metric reported under another name.

In [ ]:
from sklearn.metrics import f1_score


def run_epoch(model, loader, criterion, optimiser=None):
    training = optimiser is not None
    model.train(training)

    total_loss, predictions, actuals = 0.0, [], []

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                optimiser.zero_grad(set_to_none=True)
                loss.backward()
                optimiser.step()

            total_loss += loss.item() * len(labels)
            predictions.append(outputs.argmax(1).cpu())
            actuals.append(labels.cpu())

    predicted = torch.cat(predictions).numpy()
    actual = torch.cat(actuals).numpy()

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float((predicted == actual).mean()),
        "macro_f1": f1_score(actual, predicted, average="macro", zero_division=0),
    }


def train_model(model, loaders, epochs=30, lr=1e-3, weight_decay=1e-4,
                patience=8, parameters=None, label="model"):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimiser = torch.optim.AdamW(
        parameters if parameters is not None else model.parameters(),
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

    history, best_score, best_state, stale = [], -np.inf, None, 0
    started = time.time()

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, loaders["train"], criterion, optimiser)
        val_metrics = run_epoch(model, loaders["val"], criterion)
        scheduler.step()

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"], "val_loss": val_metrics["loss"],
            "train_f1": train_metrics["macro_f1"], "val_f1": val_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
        })

        if val_metrics["macro_f1"] > best_score:
            best_score = val_metrics["macro_f1"]
            best_state = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1

        print(f"[{label}] epoch {epoch:02d}  "
              f"train F1 {train_metrics['macro_f1']:.4f}  "
              f"val F1 {val_metrics['macro_f1']:.4f}  "
              f"val acc {val_metrics['accuracy']:.4f}")

        if stale >= patience:
            print(f"[{label}] early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    print(f"[{label}] best validation macro F1 {best_score:.4f} "
          f"in {time.time() - started:.0f}s")

    return model, pd.DataFrame(history)

In [ ]:
scratch_model, scratch_history = train_model(
    scratch_model, loaders, epochs=40, lr=1e-3, label="scratch"
)

# 8. Transfer Learning with ResNet-18

The second model reuses a ResNet-18 pretrained on ImageNet. The features that a
convolutional network learns in its early layers — oriented edges, colour opponency,
simple textures — are largely independent of the task, and with three thousand
training images they can be inherited rather than relearned.

Adaptation proceeds in two stages, which is the standard remedy for a specific
failure mode. The newly initialised classification head produces large, uninformative
gradients at the start of training; if the whole network is unfrozen immediately,
those gradients propagate backwards and destroy exactly the pretrained features the
model was chosen for.

1. **Head only.** The backbone is frozen and only the replaced final layer is trained,
   until the head produces sensible outputs.
2. **Partial fine-tuning.** `layer3` and `layer4` are unfrozen and trained at a much
   lower learning rate. The earlier blocks stay frozen because their features are
   generic and the dataset is too small to improve them.

In [ ]:
def build_transfer_model(n_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model.to(device)


transfer_model = build_transfer_model(len(class_names))

trainable = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in transfer_model.parameters())
print(f"Stage 1 — trainable {trainable:,} of {total:,} parameters")

transfer_model, transfer_stage1 = train_model(
    transfer_model, loaders, epochs=12, lr=1e-3,
    parameters=transfer_model.fc.parameters(), label="transfer-head",
)

In [ ]:
for name, parameter in transfer_model.named_parameters():
    parameter.requires_grad = name.startswith(("layer3", "layer4", "fc"))

trainable = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
print(f"Stage 2 — trainable {trainable:,} of {total:,} parameters")

transfer_model, transfer_stage2 = train_model(
    transfer_model, loaders, epochs=20, lr=1e-4, patience=6, label="transfer-finetune",
)

transfer_history = pd.concat(
    [
        transfer_stage1.assign(stage="head"),
        transfer_stage2.assign(stage="fine-tune", epoch=lambda d: d["epoch"] + len(transfer_stage1)),
    ],
    ignore_index=True,
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)

for axis, (history, title) in zip(
    axes, [(scratch_history, "Scratch CNN"), (transfer_history, "Transfer ResNet-18")]
):
    axis.plot(history["epoch"], history["train_f1"], label="train")
    axis.plot(history["epoch"], history["val_f1"], label="validation")
    axis.set_title(f"{title} — macro F1")
    axis.set_xlabel("Epoch")
    axis.legend(fontsize=8)

axes[0].set_ylabel("Macro F1")
figure.tight_layout()
va.save_figure(figure, "05_training_curves", cfg.FIGURE_DIR)
plt.show()

The gap between the training and validation curves is the quantity to read here. A
widening gap indicates that the remaining capacity is being spent on memorising
training images, and it is the signal on which the early-stopping rule acts.

# 9. Evaluation on the Held-Out Test Partition

Both models are now evaluated once, on data neither has been selected against.
Accuracy is reported alongside macro-averaged precision, recall and F1, because with
unequal class sizes accuracy alone can be high while a whole class goes undetected.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    precision_score, recall_score,
)


def evaluate(model, loader):
    model.eval()
    predictions, actuals = [], []

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            predictions.append(outputs.argmax(1).cpu())
            actuals.append(labels)

    return torch.cat(actuals).numpy(), torch.cat(predictions).numpy()


def score(name, actual, predicted):
    return {
        "Model": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Macro precision": precision_score(actual, predicted, average="macro", zero_division=0),
        "Macro recall": recall_score(actual, predicted, average="macro", zero_division=0),
        "Macro F1": f1_score(actual, predicted, average="macro", zero_division=0),
    }


y_true_scratch, y_pred_scratch = evaluate(scratch_model, loaders["test"])
y_true_transfer, y_pred_transfer = evaluate(transfer_model, loaders["test"])

results = pd.DataFrame([
    score("Scratch CNN", y_true_scratch, y_pred_scratch),
    score("Transfer ResNet-18", y_true_transfer, y_pred_transfer),
]).set_index("Model").round(4)

results.to_csv(cfg.RESULT_DIR / "test_results.csv")
results

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.4))

for axis, (actual, predicted, title) in zip(axes, [
    (y_true_scratch, y_pred_scratch, "Scratch CNN"),
    (y_true_transfer, y_pred_transfer, "Transfer ResNet-18"),
]):
    ConfusionMatrixDisplay.from_predictions(
        actual, predicted, display_labels=class_names,
        ax=axis, colorbar=False, xticks_rotation=45, normalize=None,
    )
    axis.set_title(title)

figure.tight_layout()
va.save_figure(figure, "06_confusion_matrices", cfg.FIGURE_DIR)
plt.show()

In [ ]:
print("Transfer ResNet-18 — per-class performance on the test partition\n")
print(classification_report(
    y_true_transfer, y_pred_transfer, target_names=class_names, digits=4, zero_division=0
))

# 10. Exploring a Novel Idea: What Was Leakage Control Worth?

Part 1 removed 437 near-duplicate images and constrained the partition so that no
group of possibly-shared-source images spans a boundary. That work is only justified
if it changes the number that would otherwise have been reported.

This section measures the difference directly. The identical architecture, schedule
and seed are trained a second time on the identical set of images, changing exactly
one thing: the partition ignores the leakage-risk groups and is drawn at random,
stratified only by class. This is the partition an unaudited version of this project
would have used.

The difference between the two test accuracies is not model error. It is the amount
by which an unaudited evaluation of this dataset would have overstated performance,
and it is the quantitative justification for Part 1.

In [ ]:
from sklearn.model_selection import train_test_split

leaky = manifest.copy()

train_index, holdout_index = train_test_split(
    leaky.index, test_size=0.30, stratify=leaky["label"], random_state=cfg.RANDOM_SEED
)
val_index, test_index = train_test_split(
    holdout_index, test_size=0.50,
    stratify=leaky.loc[holdout_index, "label"], random_state=cfg.RANDOM_SEED,
)

leaky.loc[train_index, "split"] = "train"
leaky.loc[val_index, "split"] = "val"
leaky.loc[test_index, "split"] = "test"

crossing = int(
    leaky.groupby("group_id")["split"].nunique().gt(1).sum()
)
print(f"Groups split across partitions by the random split: {crossing}")
print(leaky["split"].value_counts().reindex(["train", "val", "test"]).to_dict())

if crossing == 0:
    print(
        "\nNote: the random split happened to break no group, so the two "
        "partitions are equivalent in leakage terms and the comparison below "
        "measures partition variance only. Report it as such rather than as "
        "evidence about leakage."
    )
else:
    affected = int(
        leaky.groupby("group_id")["split"].transform("nunique").gt(1).sum()
    )
    print(f"Images in a broken group: {affected} ({affected / len(leaky):.1%} of the dataset)")

In [ ]:
va.set_seed(cfg.RANDOM_SEED)
leaky_loaders = build_loaders(leaky)

leaky_model = build_transfer_model(len(class_names))
leaky_model, _ = train_model(
    leaky_model, leaky_loaders, epochs=12, lr=1e-3,
    parameters=leaky_model.fc.parameters(), label="leaky-head",
)

for name, parameter in leaky_model.named_parameters():
    parameter.requires_grad = name.startswith(("layer3", "layer4", "fc"))

leaky_model, _ = train_model(
    leaky_model, leaky_loaders, epochs=20, lr=1e-4, patience=6, label="leaky-finetune",
)

y_true_leaky, y_pred_leaky = evaluate(leaky_model, leaky_loaders["test"])

In [ ]:
comparison = pd.DataFrame([
    score("Random split (leakage-risk groups broken)", y_true_leaky, y_pred_leaky),
    score("Group-aware split (audited)", y_true_transfer, y_pred_transfer),
]).set_index("Model").round(4)

comparison.loc["Overstatement"] = (
    comparison.iloc[0] - comparison.iloc[1]
).round(4)

comparison.to_csv(cfg.RESULT_DIR / "leakage_ablation.csv")
comparison

Whatever the size of the difference, the direction of the argument holds. A positive
overstatement is direct evidence that the audit prevented an inflated result. A
difference near zero is also a finding, and an honest one: it would show that VTID2
contains fewer exploitable near-duplicates than feared, and that the reported
accuracy can be trusted for a reason that has been demonstrated rather than assumed.

The result cannot be reported without one qualification. The two models are trained on
different partitions of the same images, so a small difference could arise from
partition variance alone. A rigorous estimate would repeat both arms across several
seeds and compare the distributions; with the compute available here the figure above
should be read as an indication of magnitude rather than a significance test.

# 11. What the Network Learned

A test score states how often the model is right, not why. Three views of the trained
network are examined, each corresponding to one stage of the convolutional pipeline.

## 11.1 First-layer kernels

The 64 filters of the first convolutional layer are shown directly as 7 x 7 RGB
images. In a network that has trained successfully these are recognisable oriented
edge and colour-opponent detectors; filters that remain noisy indicate capacity that
never became useful.

In [ ]:
kernels = transfer_model.conv1.weight.detach().cpu()
kernels = (kernels - kernels.min()) / (kernels.max() - kernels.min())

figure, axes = plt.subplots(4, 16, figsize=(12, 3.2))
for index, axis in enumerate(axes.flat):
    axis.imshow(kernels[index].permute(1, 2, 0))
    axis.axis("off")

figure.suptitle("First convolutional layer: learned 7x7 kernels", fontsize=11)
figure.tight_layout()
va.save_figure(figure, "07_first_layer_kernels", cfg.FIGURE_DIR)
plt.show()

## 11.2 Feature maps through the network

The same image is passed through the network and one channel is shown from each
residual stage. Activation becomes progressively sparser and coarser: early stages
respond across the whole image, later stages respond only where evidence for a class
is concentrated. This is the hierarchy the audit in Part 1 relied on when it used
`layer2` and `layer3` for spatial agreement and `layer4` only for candidate
generation.

In [ ]:
sample = manifest[manifest["split"] == "test"].iloc[0]

with Image.open(sample["path"]) as image:
    original = image.convert("RGB")
    tensor = eval_transform(original).unsqueeze(0).to(device)

recorded = {}
handles = [
    getattr(transfer_model, name).register_forward_hook(
        lambda module, inputs, output, name=name: recorded.__setitem__(name, output.detach())
    )
    for name in ("layer1", "layer2", "layer3", "layer4")
]

with torch.no_grad():
    transfer_model(tensor)

for handle in handles:
    handle.remove()

figure, axes = plt.subplots(1, 5, figsize=(13, 2.9))
axes[0].imshow(original.resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)))
axes[0].set_title(f"Input\n({cfg.CLASS_DISPLAY[sample['class_name']]})", fontsize=9)
axes[0].axis("off")

for axis, name in zip(axes[1:], ("layer1", "layer2", "layer3", "layer4")):
    maps = recorded[name][0]
    axis.imshow(maps.mean(0).cpu(), cmap="viridis")
    axis.set_title(f"{name}\n{tuple(maps.shape)}", fontsize=9)
    axis.axis("off")

figure.tight_layout()
va.save_figure(figure, "08_feature_maps", cfg.FIGURE_DIR)
plt.show()

## 11.3 Where the decision came from

Gradient-weighted class activation mapping (Selvaraju *et al.*, 2017) weights each
channel of the final feature map by the gradient of the predicted class score with
respect to that channel, then sums them. The result marks the image regions whose
features most increased the score for the predicted class.

This is the check that matters for the ethical argument in Section 13. A model that
attends to the vehicle body has learned the task; one that attends to the road
surface, a building or a watermark has learned a property of how the dataset was
collected, and would fail on imagery gathered anywhere else.

In [ ]:
def grad_cam(model, tensor, target_layer):
    """Class activation map for the predicted class at `target_layer`."""

    store = {}
    forward = target_layer.register_forward_hook(
        lambda module, inputs, output: store.__setitem__("activation", output)
    )
    backward = target_layer.register_full_backward_hook(
        lambda module, grad_in, grad_out: store.__setitem__("gradient", grad_out[0])
    )

    model.zero_grad(set_to_none=True)
    logits = model(tensor)
    predicted = int(logits.argmax(1))
    logits[0, predicted].backward()

    forward.remove()
    backward.remove()

    weights = store["gradient"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * store["activation"]).sum(dim=1)).squeeze(0)
    cam = cam / (cam.max() + 1e-8)

    return cam.detach().cpu().numpy(), predicted


examples = (
    manifest[manifest["split"] == "test"]
    .groupby("class_name", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

figure, axes = plt.subplots(2, len(examples), figsize=(2.1 * len(examples), 4.4))

for column, row in examples.iterrows():
    with Image.open(row["path"]) as image:
        picture = image.convert("RGB").resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))
    tensor = eval_transform(picture).unsqueeze(0).to(device)

    cam, predicted = grad_cam(transfer_model, tensor, transfer_model.layer4)

    axes[0, column].imshow(picture)
    axes[0, column].set_title(cfg.CLASS_DISPLAY[row["class_name"]], fontsize=9)
    axes[1, column].imshow(picture)
    axes[1, column].imshow(
        np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
            (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))),
        cmap="jet", alpha=0.45,
    )
    axes[1, column].set_title(f"predicted: {class_names[predicted]}", fontsize=8)
    axes[0, column].axis("off")
    axes[1, column].axis("off")

figure.tight_layout()
va.save_figure(figure, "09_class_activation_maps", cfg.FIGURE_DIR)
plt.show()

# 12. Discussion

*Complete this section against your own results. The prompts below identify what a
reader will look for; replace each with the figures the notebook produced.*

**Which model performed better, and was the difference worth its cost?** Compare the
two rows of the Section 9 table, and set the improvement against the fact that the
transfer model inherits ImageNet representations while the scratch model does not.
Note the training times as well as the scores.

**Which classes were confused, and is the confusion structured?** Read the confusion
matrices in the direction the errors run. Confusion between Sedan and Hatchback would
be expected — the classes differ in the treatment of the rear only, which a fixed
viewpoint may not show. Confusion involving *Other* is different in kind: that class
is defined by exclusion rather than by a shared appearance, so it has no consistent
visual signature to learn.

**Did the smaller classes suffer?** Compare the per-class recall for SUV and Hatchback
against Pickup. If the weighted loss worked, macro recall should be close to accuracy;
a large gap indicates the imbalance was not fully addressed.

**What did leakage control cost and buy?** State the Section 10 difference, and say
plainly which of the two figures should be quoted as this project's result.

**What limits these numbers?** The test partition contains roughly 650 images, so a
single-image change moves accuracy by about 0.15 percentage points. Differences
smaller than a percentage point should not be described as one model beating another.

# 13. Ethical and Societal Considerations

**Images of vehicles are not anonymous data.** A registration plate identifies a
keeper, and a photograph taken in traffic can capture faces and locations. Part 1
found evidence that some images in this dataset exist in both original and obscured
forms, which indicates that anonymisation was applied but not that it was applied
uniformly. Any onward use of the dataset should verify plate and face obscuration
across the whole collection, not on a sample.

**Vehicle type is a proxy for socio-economic status.** A model that classifies
vehicles can be composed into a system that infers something about their owners.
Deployed at a fixed camera, it produces a record of which kinds of vehicle pass a
location and when — information whose sensitivity depends entirely on what the
location is. The model is not neutral with respect to how it is deployed, and this
should be stated in any documentation accompanying it.

**Performance is not uniform, and unequal error is unequal treatment.** The classes
are unequally represented, and the per-class recall in Section 9 will differ between
them. If such a model informed tolling, access control or enforcement, the class with
the lowest recall would bear the highest rate of incorrect decisions. Reporting only
overall accuracy conceals that distribution, which is why macro-averaged metrics and
the per-class report are given here.

**The evidence base is narrow.** VTID2 is drawn from a limited range of locations,
cameras and conditions. The class activation maps in Section 11.3 show whether the
model attends to the vehicle or to its surroundings; where it attends to the
surroundings, the model has learned about the collection setting and should not be
expected to transfer. Reporting an accuracy figure without stating that limitation
would overstate what has been demonstrated.

**Provenance and reuse.** The dataset carries the licence and consent basis under
which it was collected. Those terms constrain redistribution and any commercial reuse
of models trained on it, and they apply to the derived artefacts in this repository as
well as to the images.

# 14. Reflection on Learning

*Written in the first person, as required for the portfolio. Replace the prompts with
your own account.*

**On where the difficulty actually was.** Most of my time on this project went to
deciding which images could be trusted, not to designing networks. My first attempt at
duplicate detection searched for a similarity threshold and did not converge, because
I was asking one number to decide both what to delete and what to keep together — two
decisions with opposite costs. Separating them resolved in an afternoon a problem I
had spent much longer on.

**On using a network as an instrument rather than a classifier.** Before this project I
thought of a CNN as something that outputs a class. Using ResNet-18 as a measuring
device, and finding that its final layer answers "same kind of vehicle?" while its
middle layers answer "same photograph?", changed how I read the idea that
convolutional features are hierarchical. It stopped being a diagram and became
something I had to depend on.

**On negative results.** Establishing that the deep embedding *cannot* separate the
cases was the most useful finding in Part 1, and I initially treated it as a failure
to be hidden. Reported as a measurement, it is what justifies everything built on top
of it.

**On what I would do differently.** I would define the labelled reference set and the
selection objective before computing any similarity, rather than after. Almost all of
the wasted effort came from evaluating candidate rules by eye, one screen of images at
a time, with no criterion for stopping.

**On what I would do next.** I would repeat the Section 10 comparison across several
seeds to separate the leakage effect from partition variance, and test the model on
vehicle images from a different source to check whether it generalises beyond this
dataset's collection conditions.

# 15. Conclusion

*Complete against your results.*

This project developed convolutional classifiers for five vehicle types from the
VTID2 image dataset. Its distinguishing feature is not the architectures, which are
standard, but the evidence assembled before any model was trained: a documented audit
that identified byte-identical and near-duplicate images, separated deletion from
partition control as two decisions with different error costs, calibrated the deletion
rule against a manually labelled reference set at a stated operating point, and
produced a partition in which no group of possibly-shared-source images spans a
boundary.

*State the final test accuracy and macro F1 of the selected model, the difference
measured in Section 10, and which figure you are putting forward as this project's
result — together with the reason it can be believed.*

# References

Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K. and Fei-Fei, L. (2009) 'ImageNet: a
large-scale hierarchical image database', *IEEE Conference on Computer Vision and
Pattern Recognition*, pp. 248-255.

He, K., Zhang, X., Ren, S. and Sun, J. (2016) 'Deep residual learning for image
recognition', *IEEE Conference on Computer Vision and Pattern Recognition*,
pp. 770-778.

Ioffe, S. and Szegedy, C. (2015) 'Batch normalization: accelerating deep network
training by reducing internal covariate shift', *Proceedings of the 32nd International
Conference on Machine Learning*, pp. 448-456.

Kingma, D.P. and Ba, J. (2015) 'Adam: a method for stochastic optimization',
*International Conference on Learning Representations*.

Lin, M., Chen, Q. and Yan, S. (2014) 'Network in network', *International Conference
on Learning Representations*.

Loshchilov, I. and Hutter, F. (2019) 'Decoupled weight decay regularization',
*International Conference on Learning Representations*.

Selvaraju, R.R., Cogswell, M., Das, A., Vedantam, R., Parikh, D. and Batra, D. (2017)
'Grad-CAM: visual explanations from deep networks via gradient-based localization',
*IEEE International Conference on Computer Vision*, pp. 618-626.

Shorten, C. and Khoshgoftaar, T.M. (2019) 'A survey on image data augmentation for
deep learning', *Journal of Big Data*, 6(1), 60.

Simonyan, K. and Zisserman, A. (2015) 'Very deep convolutional networks for
large-scale image recognition', *International Conference on Learning
Representations*.

Yosinski, J., Clune, J., Bengio, Y. and Lipson, H. (2014) 'How transferable are
features in deep neural networks?', *Advances in Neural Information Processing
Systems*, 27, pp. 3320-3328.

**Dataset.** VTID2 (Vehicle Type Image Dataset 2). *Insert the full citation for the
copy of the dataset you obtained before submission.*